来到最终章啦，我们怎么把模型赋予智慧呢，现在我们有了一个种子，需要用人类的知识将它浇灌为参天大树，下面会介绍几个关键概念，这些概念不仅限于Transformer模型，其他的深度学习模型也是适用的，后续如果出了新的模型，你自己手写完训练的时候也用的上，（模型结构大变天我说了就不算了）～


损失函数与优化器
想象你在一个漆黑的山谷里，目标是走到最低点（下山）：
损失函数（Loss Function）：就是你脚下的 "地形图"，它告诉你 "你现在离谷底还有多远"，也就是模型预测结果和真实答案之间的差距有多大。
优化器（Optimizer）：就是你的 "下山策略"，它决定你每一步往哪走、步子迈多大，帮你一步步走到谷底。
训练模型的本质：不断调整模型参数，让损失函数的值越来越小。

不同的模型会有不同的偏好，举例说明
| 环节         | CNN 场景                                                                 | Transformer 场景                                                                                          |
| ------------ | ------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------- |
| **典型任务** | 图像分类、目标检测                                                       | 机器翻译、大语言模型（GPT/BERT）                                                                         |
| **损失函数** | 分类用 Cross-Entropy；检测用 Cross-Entropy + Smooth L1（定位框回归）     | 语言模型用 Cross-Entropy（预测下一个 token）；BERT 还有 MLM Loss（完形填空）                              |
| **优化器**   | 经典用 SGD + Momentum（配合学习率衰减），也常用 Adam                     | 几乎统一用 AdamW                                                                                          |
| **学习率策略** | StepLR / CosineAnnealing，训练中逐步降低                               | Warmup + 衰减：先从极小值线性增大（热身），再逐步减小。Transformer 对学习率非常敏感，没有 Warmup 容易训练崩溃 |



一张图总结训练过程
```
输入数据 → 模型(CNN/Transformer) → 预测结果
                                        ↓
                                   损失函数 ← 真实标签
                                        ↓
                                    损失值(一个数字)
                                        ↓
                                   反向传播(算梯度)
                                        ↓
                                   优化器(更新参数)
                                        ↓
                                   回到第一步，循环
```

还记得我们在处理输入时，添加了Padding，那计算loss的时候，就不应该考虑无意义Padding的loss，避免模型学习一些无意义的模式。

In [ ]:
import torch
from torch import nn
import torch.optim as optim

# 定一个损失函数，交叉熵损失函数，ignore_index=0表示在计算损失时忽略标签为0的样本，
criterion = nn.CrossEntropyLoss(ignore_index=0)
## 原文中使用的是Adam优化器，
# 为了简单，我们使用默认的学习率0.0001，beta1=0.9, beta2=0.98, eps=1e-9，这几个参数的含义不用特别理解：

optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

model.train()

定义好小驴子前面的萝卜，我们就要开始训练了，因为我们是有答案的，所以我们使用的是有监督训练，并且是有监督训练中的 Teacher Forcing。

训练时，Decoder的输入不是模型自己的预测，而是真实的目标序列（就像老师手把手教）。

不用 Teacher Forcing（自回归 / Free Running）

模型自己的输出作为下一步的输入：
```
时间步    输入（模型自己的上一步输出）    模型预测    正确答案
──────────────────────────────────────────────────────────
  t=1        <start>                      "我"        "我"   ✅
  t=2        "我"                         "恨"        "爱"   ❌ ← 预测错了！
  t=3        "恨"（错误被带入！）           "狗"        "猫"   ❌ ← 错上加错！
```
问题：一步错，步步错，错误像滚雪球一样越滚越大。这叫 error accumulation（误差累积）。
用 Teacher Forcing

不管模型上一步输出了什么，都把正确答案塞给它作为下一步输入：
```
时间步    输入（强制喂入正确答案）     模型预测    正确答案
──────────────────────────────────────────────────────────
  t=1        <start>                   "我"        "我"   ✅
  t=2        "我"  ← 正确答案          "恨"        "爱"   ❌（但没关系！）
  t=3        "爱"  ← 正确答案          "猫"        "猫"   ✅（不受上一步影响）
```
关键点：即使 t=2 预测错了，t=3 的输入仍然是正确的 "爱"，不会被带偏。模型可以更快、更稳定地学习。

| 场景       | Free Running                                                                 | Teacher Forcing                                                                 |
|------------|------------------------------------------------------------------------------|---------------------------------------------------------------------------------|
| 学做菜     | 你上一步切错了葱（切成了块），就用错的葱块继续炒，最后出来的菜味道全错       | 老师看到你切错了，直接把切好的葱丝递给你，让你继续用正确的食材炒下去             |
| 写作文     | 你第一句跑题了，后面的句子就着跑题的句子继续写，越写越偏                     | 老师把你跑题的第一句改正，让你基于正确的上文继续写                               |
| **本质**   | 模型靠自己过去的输出 "自学"，容易一错到底                                   | 老师每一步都给正确答案做参照，训练效率高                                         |




In [ ]:
# 训练的epochs， 整个训练集被模型 "看" 多少遍。1 epoch = 所有数据过一次前向 + 反向传播
epochs = 100

for epoch in range(epochs):
    epoch_loss = 0.0

    for src, tgt in loader:
        # src : (Batch, Src_len)
        # tgt : (Batch, Tgt_len) -> 包含 <sos> ... <eos>
        src, tgt = src.to(device), tgt.to(device)
        # 核心：构造Decoder的输入和标签
        # Decoder Input，去掉最后一个元素
        tgt_input = tgt[:, :-1]
        # Decoder Label，去掉第一个元素
        tgt_label = tgt[:, 1:]

        # 1. 清空梯度
        optimizer.zero_grad()

        # 2.前向传播
        output = model(src, tgt_input)

        # output shape : (Batch, Tgt_len-1, Vocab_size)
        # tgt_label shape : (Batch, Tgt_len-1)

        # 3.计算损失
        # CrossEntropyLoss 需要（N,C）和 （N）的形状，所以，要做一次打平
        loss = criterion(output.contiguous().view(-1, len(tar_vocab)), tgt_label.contiguous().view(-1))

        # 4.反向传播
        loss.backward()

        # 5.更新参数
        optimizer.step()

        epoch_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        # 输出下当前的损失，看看模型的学习进度，epoch_loss/len(loader)是平均每个batch的损失
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(loader):.4f}')

print("训练完成！")

有聪明的小朋友可能会问epochs为什么大于1，数据学习一一遍不就行了，为啥要翻来覆去让模型学习同一份数据？

类比下你学习和备考的时候，不会只看一遍课本就上考场。而是：
第一遍：粗略理解概念（= 模型第一轮训练，loss 很高）
反复刷题：做大量练习题，对答案，纠错（= 多轮 epoch 训练，反向传播调参数）
模拟考试：用从没见过的题目检验自己（= 验证集 / 测试集评估）

再说一些理论的内容：不同 epoch 学到的东西不一样

训练过程可以粗略分为几个阶段：
早期 epoch：模型学习数据中最显著、最粗粒度的模式（比如 "猫有耳朵"）
中期 epoch：开始捕获更细微的特征和边界情况（比如 "猫耳的不同形状"）
后期 epoch：精调决策边界，优化难以区分的样本
每一轮遍历，模型都是在不同的参数状态下重新审视数据，因此即使是相同的样本，在不同 epoch 中提供的梯度信号也是不同的。

但也不能看太多遍 —— 过拟合风险

多轮遍历也有代价。遍历次数过多，模型可能从 "学习规律" 变成 **"记忆数据"，即过拟合（overfitting）**。因此实践中需要：
用 验证集（validation set） 监控泛化性能
使用 Early Stopping：验证集性能不再提升时停止训练
配合 正则化（Dropout、Weight Decay 等）抑制过拟合

先搞懂 CrossEntropyLoss 是干嘛的

CrossEntropyLoss 是深度学习中一个非常常用的损失函数（Loss Function），主要用在分类任务中。它的作用就是：衡量模型的预测结果和真实答案之间差了多远。
什么是 (N, C) 和 (N)？

这是在说数据的形状（shape），你可以把它想象成 Excel 表格的行和列：

| 符号 | 含义 | 类比 |
|------|------|------|
| N | 样本数量（有多少条数据） | Excel 表格的行数 |
| C | 类别数量（一共有几种分类） | Excel 表格的列数 |

模型的预测结果：形状是 (N, C)

这是一个二维表格。比如你在做一个 "猫🐱/ 狗🐶/ 鸟🐦" 三分类任务，有 4 张图片：

|        | 猫的概率 | 狗的概率 | 鸟的概率 |
|--------|----------|----------|----------|
| 图片 1 | 0.8      | 0.1      | 0.1      |
| 图片 2 | 0.2      | 0.7      | 0.1      |
| 图片 3 | 0.1      | 0.1      | 0.8      |
| 图片 4 | 0.3      | 0.6      | 0.1      |

这个表格就是 (4, 3) 的形状 —— 4 个样本，3 个类别。
真实答案：形状是 (N)

这是一个一维列表，只告诉你每张图片正确答案是第几类：
`[0, 1, 2, 1]`




# CrossEntropyLoss 展平计算详解

这一步的核心问题是：PyTorch 的 `CrossEntropyLoss` 对输入形状有要求，而 Transformer 的输出是 3D 张量，所以需要展平（reshape）。

---

## 1. 先明确各张量的含义

假设一个极简的 case：

| 参数 | 值 | 含义 |
|---|---|---|
| Batch | 2 | 2 个句子 |
| Seq_Len_Out | 3 | 每个句子预测 3 个 token |
| Vocab_Size | 5 | 词表大小为 5（只有 5 个词） |

---

## 2. 模型输出 output

形状 `(2, 3, 5)` —— 对每个位置，模型给出 5 个词各自的「未归一化得分」（logits）。

```
句子0:
  位置0: [2.0, 1.0, 0.1, 0.5, 0.3]   → 模型觉得词0最可能
  位置1: [0.1, 0.2, 3.0, 0.1, 0.4]   → 模型觉得词2最可能
  位置2: [0.3, 0.1, 0.2, 0.1, 2.5]   → 模型觉得词4最可能

句子1:
  位置0: [0.1, 2.5, 0.3, 0.1, 0.2]   → 模型觉得词1最可能
  位置1: [1.5, 0.2, 0.1, 2.0, 0.3]   → 模型觉得词3最可能
  位置2: [0.2, 0.1, 0.3, 0.1, 3.0]   → 模型觉得词4最可能
```

---

## 3. 真实标签 tgt_y

形状 `(2, 3)` —— 每个位置上「正确答案」的词索引。

```
句子0: [0, 2, 4]    → 正确的词分别是 词0、词2、词4
句子1: [1, 3, 4]    → 正确的词分别是 词1、词3、词4
```

---

## 4. 为什么要展平？

`CrossEntropyLoss` 要求输入是：

- **预测**：`(N, C)` —— N 个样本，C 个类别
- **标签**：`(N,)` —— N 个样本的正确类别索引

但我们的数据是 3D 的。所以要把 `Batch` 和 `Seq_Len_Out` 合并成一个维度 N：

```
N = Batch × Seq_Len_Out = 2 × 3 = 6
```

**展平 output** `.view(-1, 5)` → 形状变成 `(6, 5)`：

```
样本0 (句子0-位置0): [2.0, 1.0, 0.1, 0.5, 0.3]
样本1 (句子0-位置1): [0.1, 0.2, 3.0, 0.1, 0.4]
样本2 (句子0-位置2): [0.3, 0.1, 0.2, 0.1, 2.5]
样本3 (句子1-位置0): [0.1, 2.5, 0.3, 0.1, 0.2]
样本4 (句子1-位置1): [1.5, 0.2, 0.1, 2.0, 0.3]
样本5 (句子1-位置2): [0.2, 0.1, 0.3, 0.1, 3.0]
```

**展平 tgt_y** `.view(-1)` → 形状变成 `(6,)`：

```
[0, 2, 4, 1, 3, 4]
```

---

## 5. CrossEntropyLoss 对每个样本怎么算？

以 **样本 0** 为例，logits = `[2.0, 1.0, 0.1, 0.5, 0.3]`，正确答案 = `0`（词 0）。

### Step A：Softmax

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

```
e^2.0 = 7.389,  e^1.0 = 2.718,  e^0.1 = 1.105,  e^0.5 = 1.649,  e^0.3 = 1.350
总和 = 14.211

p = [0.520, 0.191, 0.078, 0.116, 0.095]
```

### Step B：取正确类别的概率，算负对数

```
正确答案是词 0，对应概率 = 0.520
loss_0 = -log(0.520) = 0.654
```

### 对每个样本重复同样过程：

| 样本 | logits | 正确词 | 正确词概率 | 单样本 loss |
|---|---|---|---|---|
| 0 | [2.0, 1.0, 0.1, 0.5, 0.3] | 0 | 0.520 | 0.654 |
| 1 | [0.1, 0.2, 3.0, 0.1, 0.4] | 2 | 0.576 | 0.551 |
| 2 | [0.3, 0.1, 0.2, 0.1, 2.5] | 4 | 0.565 | 0.572 |
| 3 | [0.1, 2.5, 0.3, 0.1, 0.2] | 1 | 0.576 | 0.551 |
| 4 | [1.5, 0.2, 0.1, 2.0, 0.3] | 3 | 0.369 | 0.997 |
| 5 | [0.2, 0.1, 0.3, 0.1, 3.0] | 4 | 0.620 | 0.478 |

### 最终 loss = 所有样本的平均值（默认 `reduction='mean'`）：

$$
loss = \frac{0.654 + 0.551 + 0.572 + 0.551 + 0.997 + 0.478}{6} = 0.634
$$

---

## 一句话总结

> 展平的本质就是：把「2 个句子 × 3 个位置」拉成「6 个独立的分类问题」，每个问题都是从 5 个词里选 1 个正确的，然后用交叉熵算 loss 再取平均。


那为什么序列翻译任务是一个分类任务呢

序列翻译（如机器翻译）本质上是一个逐步的分类任务，原因如下：
核心逻辑

在 seq2seq 模型（如 Transformer）的解码阶段，每一个时间步（timestep）的任务是：从整个目标语言的词汇表（vocabulary）中，选出一个概率最高的词（token）作为当前位置的输出。
这个 "从 V 个候选词中选一个" 的操作，本质上就是一个 V 分类问题（V = 词汇表大小，通常几万量级）。
具体机制

Decoder 的最后一层会输出一个维度为 V 的 logits 向量。
对这个向量做 softmax，得到在整个词汇表上的概率分布。
选取概率最大的那个 token（或用 beam search 等策略），作为当前时间步的预测。
训练时用的损失函数是 交叉熵（Cross-Entropy Loss）—— 标准的分类损失。
一句话总结

翻译任务在序列层面是生成任务，但拆解到每个时间步，就是在词汇表上做一次多分类。整个翻译过程 = 连续执行 T 次分类（T 为目标序列长度）。

Transformer 训练循环：每一步的权重与 Loss 变化详解

下面我将逐步拆解这个训练循环，详细说明每一步中 **模型权重（weights）** 和 Loss 的状态变化。
整体流程概览

```
optimizer.zero_grad() → forward → loss计算 → backward → optimizer.step()
```

每个 mini-batch 都遵循这个 5 步循环。我们逐步分析：

## Step 1: optimizer.zero_grad() — 清空梯度

| 变量 | 变化 |
|------|------|
| 权重 W | 不变。权重值保持上一步 optimizer.step() 更新后的状态 |
| 梯度 W.grad | 归零。所有参数的 .grad 属性被设为 0（或 None → 0） |
| Loss | 尚未计算，此时无 Loss 值 |

为什么需要这一步？ PyTorch 默认会累加梯度（grad += new_grad），如果不清零，本次 batch 的梯度会和上一次 batch 的梯度混在一起，导致更新方向错误。

## Step 2: output = model(src, tgt_input) — 前向传播

| 变量 | 变化 |
|------|------|
| 权重 W | 不变。前向传播只是用当前权重做矩阵运算，不修改权重 |
| 梯度 W.grad | 不变。仍为 0（前向传播不计算梯度） |
| 中间激活值 | 生成并缓存。每一层的输出（attention scores、FFN 中间值等）被保存在计算图中，供反向传播使用 |
| output | 新生成。shape 为 (Batch, Seq_Len_Out, Vocab_Size)，是模型对每个位置预测的词汇概率分布（logits） |

详细过程：

```
src → Encoder → memory
                  ↓
tgt_input → Decoder(memory) → Linear → output (logits)
```

数据流经模型的每一层：
- Encoder：src 经过 Embedding → Positional Encoding → N 层 (Self-Attention → FFN)
- Decoder：tgt_input 经过 Embedding → Positional Encoding → N 层 (Masked Self-Attention → Cross-Attention → FFN)
- 最终线性层：将 Decoder 输出映射到词汇表大小

每一层的运算都是：output = f(input, W)，权重 W 参与计算但自身不变。

## Step 3: loss = criterion(output.view(-1, V), tgt_y.view(-1)) — 计算 Loss

| 变量 | 变化 |
|------|------|
| 权重 W | 不变 |
| 梯度 W.grad | 不变，仍为 0 |
| Loss | 新计算得出，是一个标量 tensor，带有完整的计算图 |

具体计算：

```python
# 展平操作
output: (Batch, Seq_Len, Vocab_Size) → (Batch × Seq_Len, Vocab_Size)
tgt_y:  (Batch, Seq_Len)             → (Batch × Seq_Len,)

# CrossEntropyLoss 内部做了两件事：
# 1. LogSoftmax: 将 logits 转为 log 概率
# 2. NLLLoss: 取出正确标签位置的 log 概率，取负平均
```

Loss 值的含义：
- 训练初期：权重随机初始化，模型预测接近均匀分布，Loss ≈ ln(Vocab_Size)。例如词汇表大小 10000 时，初始 Loss ≈ 9.21
- 训练中期：模型逐渐学到模式，Loss 持续下降
- 训练后期：Loss 趋于收敛，下降速度变慢

## Step 4: loss.backward() — 反向传播

| 变量 | 变化 |
|------|------|
| 权重 W | 不变！反向传播只计算梯度，不更新权重 |
| 梯度 W.grad | 从 0 变为具体值。每个参数都获得了梯度 ∂Loss/∂W |
| Loss | Loss 值不变（仍然是同一个标量） |
| 计算图 | 被释放（默认 retain_graph=False），中间激活值的内存被回收 |

详细过程（链式法则反向传递）：

```
Loss
 ↓ ∂Loss/∂output
Linear 层 (Decoder 最后的映射层)
 ↓ ∂Loss/∂decoder_output
Decoder 第 N 层 FFN
 ↓
Decoder 第 N 层 Cross-Attention
 ↓
Decoder 第 N 层 Masked Self-Attention
 ↓
...逐层向前回传...
 ↓
Decoder 第 1 层
 ↓
Decoder Embedding
 ↓ (同时向 Encoder 回传)
Encoder 第 N 层
 ↓
...
Encoder 第 1 层
 ↓
Encoder Embedding
```

每一层在反向传播时：
- 接收来自上游的梯度 ∂Loss/∂output
- 利用前向传播缓存的激活值，计算两样东西：
  - 参数梯度 ∂Loss/∂W → 累加到 W.grad
  - 输入梯度 ∂Loss/∂input → 传给下一层（更靠近输入的层）

反向传播结束后，模型中每一个可训练参数（Embedding 矩阵、Q/K/V 权重、FFN 权重、LayerNorm 参数、最终 Linear 层等）都拥有了对应的梯度值。

## Step 5: optimizer.step() — 更新参数

| 变量 | 变化 |
|------|------|
| 权重 W | 被更新！这是唯一修改权重的步骤 |
| 梯度 W.grad | 不变（值还在，直到下一次 zero_grad() 清零） |
| Loss | 此时的 Loss 对应的是更新前的权重，新权重的 Loss 要到下一次前向传播才知道 |

更新公式（以 Adam 为例）：

```python
# Adam 优化器内部维护了两个状态变量（每个参数各一份）：
# m: 一阶矩估计（梯度的指数移动平均）
# v: 二阶矩估计（梯度平方的指数移动平均）

m = β₁ * m + (1 - β₁) * grad           # 更新动量
v = β₂ * v + (1 - β₂) * grad²          # 更新二阶矩
m̂ = m / (1 - β₁ᵗ)                      # 偏差校正
v̂ = v / (1 - β₂ᵗ)                      # 偏差校正

W_new = W_old - lr * m̂ / (√v̂ + ε)      # 权重更新
```

典型的权重变化幅度（使用默认 Adam，lr=1e-4）：

| 训练阶段 | 单步权重变化量级 | 说明 |
|----------|------------------|------|
| 初期 | ~10⁻⁴ | 梯度大，学习率生效，权重变化明显 |
| 中期 | ~10⁻⁵ | 梯度变小，Adam 的自适应学习率也在调整 |
| 后期 | ~10⁻⁶ 或更小 | 接近收敛，权重几乎不再变化 |

## epoch_loss += loss.item() — 累积 Loss

| 变量 | 变化 |
|------|------|
| 权重 W | 不变（这行只是记录数值） |
| Loss tensor | .item() 取出 Python float，切断计算图引用，允许 GC 回收 Loss tensor 占用的显存 |
| epoch_loss | 累加当前 batch 的 loss 标量值 |

## 完整的一个 Epoch 视角

```
Epoch 开始
│
├── Batch 1: zero_grad → forward → loss → backward → step
│   W: W₀ → W₀ → W₀ → W₀ → W₁        Loss: ? → ? → L₁ → L₁ → L₁
│
├── Batch 2: zero_grad → forward → loss → backward → step
│   W: W₁ → W₁ → W₁ → W₁ → W₂        Loss: ? → ? → L₂ → L₂ → L₂
│
├── Batch 3: zero_grad → forward → loss → backward → step
│   W: W₂ → W₂ → W₂ → W₂ → W₃        Loss: ? → ? → L₃ → L₃ → L₃
│
├── ...
│
└── Batch N: zero_grad → forward → loss → backward → step
    W: Wₙ₋₁ → ... → Wₙ                 Loss: Lₙ
    
    epoch_loss = L₁ + L₂ + L₃ + ... + Lₙ
    打印: epoch_loss / N
```

跨 Epoch 的 Loss 变化趋势

```
Epoch  1 | Loss: 8.9234    ← 接近 ln(vocab_size)，模型在"瞎猜"
Epoch 10 | Loss: 4.2156    ← 快速下降阶段
Epoch 20 | Loss: 1.8923    ← 学到了主要模式
Epoch 30 | Loss: 0.6741    ← 精细调整阶段
Epoch 40 | Loss: 0.2103    ← 接近收敛
Epoch 50 | Loss: 0.0847    ← 在训练集上拟合良好
```

## 关键总结

| 步骤 | 权重 W | 梯度 W.grad | Loss | 内存 / 显存 |
|------|--------|-------------|------|-------------|
| zero_grad() | 不变 | 归零 | 无 | 梯度内存清零 |
| forward | 不变 | 0 | 无 | 激活值缓存增加 |
| loss = criterion(...) | 不变 | 0 | 计算得出 | Loss tensor + 计算图 |
| backward() | 不变 | 0 → 具体值 | 不变 | 计算图释放，梯度内存占用 |
| step() | 更新！ | 不变 | 过时的旧值 | 优化器状态更新 |

核心认知： 前向传播、Loss 计算、反向传播三步都不修改权重，它们只是为了计算出 "权重该往哪个方向调整、调整多少"（即梯度）。只有最后的 optimizer.step() 才真正修改权重。

训练好的模型有答案，但是模型在真的回答你问题的时候，他是没有标准答案的，怎么办呢？

1. 先把源句子编码成Memory
2. 给Decoder输入<sos>
3. 模型根据Memory和<sos>生成第一个词
4. 模型根据Memory和第一个词生成第二个词
5. ...
6. 模型生成<eos>，结束



In [ ]:
def greedy_decode(model, src, src_mask, max_len, start_symbol):

    src = src.to(device)
    src_mask = src_mask.to(device)

    # 1.这就是我们为了推理拆分出来的函数的意义， Encoder只需要计算一次。
    memory = model.encode(src, src_mask)

    # 2.初始化 Decoder输入，只有<sos>
    # shape (1, 1)假设一次翻译一个句子
    ys = torch.ones(1,1).fill_(start_symbol).type(torch.long).to(device)

    for i in range(max_len-1):
        # 准备mask，避免看到未来信息，虽然推理没有未来，为了格式统一
        tgt_mask = make_subsequent_mask(ys.size(1)).to(device)
        
        # 3. 模型根据Memory和ys生成下一个词
        # shape (1, i+1)
        out = model.decode(ys, memory, src_mask, tgt_mask)
        
        # 4. out shape(1, seq_len, vocab_size) 因为 Decoder 内部已经包含了 projection 层
        # 我们只关心最后的时间步的输出
        prob = out[:, -1, :]
        
        # 5. 取概率最大的词
        _, next_word = torch.max(prob, dim=1) # 假如获取到概率最大的词的索引是tensor([376])
        next_word = next_word.item() # 转换为int 376
        
        # 6. 把这个词添加到ys后面
        #  torch.ones(1,1)          →  tensor([[1.0]])
        #    .type_as(src.data)     →  tensor([[1]])       # 类型+设备对齐
        #    .fill_(next_word)      →  tensor([[56]])       # 填入预测词
        #  torch.cat([ys, ...], 1)  →  ys 末尾多了一个词
        ys = torch.cat([ys, torch.ones(1,1).type_as(src.data).fill_(next_word)], dim=1)
        
        # 7. 如果是<eos>，就结束
        if next_word == tar_vocab['<eos>']:
            break
            
    return ys


#--- 封装一个翻译函数 ---
def translate(sentence):
    model.eval()  # 切换到评估模式(关闭Dropout等)
    
    # 文本转索引
    src_indices = [src_vocab[w] for w in sentence.split()]
    src_tensor = torch.tensor(src_indices).unsqueeze(0).to(device)

    # 2. 生成掩码
    src_mask = make_pad_mask(src_tensor, src_tensor, 0)
    
    # 3. 调用模型进行翻译
    out_tensor = greedy_decode(model, src_tensor, src_mask, max_len=100, start_symbol=tar_vocab['<sos>'])
    
    # 4. 把索引序列转换为句子
    out_indices = out_tensor.squeeze().tolist()  # shape (1, seq_len) -> (seq_len)
    translation = []
    for idx in out_indices:
        if idx == tar_vocab['<sos>']: continue  # 跳过<sos>
        if idx == tar_vocab['<eos>']: break  # 遇到<eos>结束
        translation.append(idx2tar[idx])
    
    return "".join(translation)

# --- 见证奇迹的时刻 ---
print("\n===测试模型翻译能力===")

test_sentences = [
    "I love deep learning",
    "Transformer is all you need",
    "hello world"
]

for s in test_sentences:
    translation = translate(s)
    print(f"原文: {s}  -->  翻译: {translation}")
    print("-" * 20)

可视化注意力 
还记得之前 forward返回的 attn_map吗，现在能展示模型在翻译某个中文词时，在这个瞬间“盯着”哪个英文词看。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_attention(sentence, translation, attention_weights):
    """
    sentence: 原文句子
    translation: 翻译结果
    attention_weights: 注意力权重矩阵，shape (tgt_len, src_len)
    """
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 设置中文字体
    plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(attention_weights, cmap='viridis', xticklabels=sentence.split(), yticklabels=list(translation))
    plt.xlabel('Source Words')
    plt.ylabel('Target Words')
    plt.title('Attention Weights')
    plt.show()

# --- 获取 Attention Map 并绘图 ---
# 手动跑一次forward 来拿 attention

model.eval()
src_text = "I love deep learning"
src_indices = torch.tensor([src_vocab[w] for w in src_text.split()]).unsqueeze(0).to(device)
src_mask = make_pad_mask(src_indices, src_indices, 0)


# 假设我们已经翻译好了得到 tgt_indices (这里简化直接模拟解码后的结果)
# 实际应该修改 greedy_decode 让它把 attention 也返回出来
# 这里演示原理：
enc_out = model.encode(src_indices, src_mask)
# 假设 Decoder 输入是 "我 爱 深度 学习"
tgt_indices = torch.tensor([[1,
                             tar_vocab['我'],
                             tar_vocab['爱'],
                             tar_vocab['深'],
                             tar_vocab['度'],
                             tar_vocab['学'],
                             tar_vocab['习']]]).to(device)
tgt_mask = make_subsequent_mask(tgt_indices.size(1)).to(device)

# 跑一次 decoder，注意这里直接调用内部的 _decoder 来获取 attention map
output, attn_weights = model._decoder(model.tgt_embedding(tgt_indices), enc_out, src_mask, tgt_mask)

# attn_weights shape: (Batch, Heads, Tgt_Len, Src_Len)
# 我们取第一个样本，平均所有头的注意力，或者只看第一个头
attn_avg = attn_weights[0].mean(dim=0).detach().cpu().numpy()  # (Tgt_Len, Src_Len)

# 去掉 <sos> 的影响方便看图
plot_attention(src_text, "<sos>我爱深度学习", attn_avg[1:])

至此，你已经从零开始构建并训练了一个Transformer模型，并且完成了一次翻译任务，虽然这个模型很小，数据集也很小，但是原理和ChatGPT 和 Claude模型一样，你把模型放大，数据放大，模型训练时间变长，就会得到一个非常强大的模型。

用一个现代诗结尾：

Embedding 把世界万物编织成向量，
让语言有了形状，让含义有了方向。
Attention 在向量的海洋中捕捞关联，
让每个词回望全文，找到自己的锚点。
Training 让模型一次次预测下一个 Token，
在错误与修正之间，逼近语言的本能。
Loss 是黑暗中唯一的灯塔，
梯度沿着它的方向，一步步下山。
Scaling 是一个朴素的信仰 ——
参数再多一些，数据再大一些，智能就涌现了。
Inference 是训练之后的独白，
模型终于开口，一个 Token 接一个 Token，
像孩子学会造句，
像河流学会入海。
我们用矩阵乘法搭建了一座巴别塔，
它不通天，
但它开始理解。

课后作业，可以去Kaggle或者开源数据集下载一些 中英，中日，或者你喜欢的数据集， 然后把模型扩大，看下能不能训练一个自己的翻译模型。

可能的番外篇： 大模型怎么微调？ 可能会写一篇， 改造下，改成MOE模型也有可能写一篇。

然后会开启一个更上层系列就是，利用各种工程实践，写一个你自己的Agent，包括，记忆，context管理，工具调用等等。

还想开启一个更底层的系列，用一些代码模拟GPU，然后让你知道Transformer模型在GPU上是如何训练的。

